In [1]:
import os

if os.path.exists("NovaS.pdf"):
    print("Successfully found the file.")
else:
    print("File not found.")

Successfully found the file.


In [2]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import os
from dotenv import load_dotenv
load_dotenv()

True

##### STEP-0: LOAD THE PDF INTO THE TEXT FORMAT

In [3]:
text_data = PyPDFLoader("NovaS.pdf").load()

for page in text_data:
    page.metadata["source"] = "NovaS.pdf"
    
text_data

[Document(metadata={'source': 'NovaS.pdf', 'page': 0}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future. At the \nbeginning, the company did not have large investments or a big office. Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight. Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by delivering real value to \ntheir clients. Most of the early work involved helping small companies understand their \nexisting data and use simple reporting solutions to make better business decisions. \nDuring the first year of operations, the 

##### STEP-1: CREATING CHUNKS OF THE TEXT DATA

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = splitter.split_documents(text_data)
chunks
len(chunks)

101

##### STEP-2&3: CREATING EMBEDDINGS OF THE CHUNKS & STORING THEM IN VECTOR DB 

In [5]:
from langchain_community.vectorstores import Chroma


In [6]:
%pip install --upgrade protobuf chromadb --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-http 1.30.0 requires opentelemetry-exporter-otlp-proto-common==1.30.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.30.0 requires opentelemetry-proto==1.30.0, but you have opentelemetry-proto 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.30.0 requires opentelemetry-sdk~=1.30.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [6]:
import chromadb
import chromadb.config
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
chroma_db = Chroma.from_documents(chunks, embed_model, persist_directory="./chroma_db")



#### STEP-4: CONNECTION & RETRIEVAL OF CHROMA DB

In [7]:
chroma_db_con = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)


/var/folders/c4/y4y568g51bsbt1rt9m950y3c0000gn/T/ipykernel_43873/602062922.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)


In [8]:
chroma_db_con.similarity_search("when NovaSphere technologies was founded?", k=3)

[Document(metadata={'source': 'NovaS.pdf', 'page': 0}, page_content='In 2018, NovaSphere Technologies started getting more attention in the local technology'),
 Document(metadata={'source': 'NovaS.pdf', 'page': 2}, page_content='Today, NovaSphere Technologies is considered a reliable organization that provides data'),
 Document(metadata={'page': 2, 'source': 'NovaS.pdf'}, page_content='organizations use their data, and NovaSphere Technologies wants to play an important role')]

In [9]:
chroma_db_con.similarity_search("By 2019, how many employees were there?", k=3)


[Document(metadata={'source': 'NovaS.pdf', 'page': 0}, page_content='By the beginning of 2019, the organization had grown to more than fifteen employees. This'),
 Document(metadata={'page': 1, 'source': 'NovaS.pdf'}, page_content='During 2023, the organization experienced steady growth. The number of employees'),
 Document(metadata={'page': 1, 'source': 'NovaS.pdf'}, page_content='number of employees increased again, and the company also started hiring people who')]

#### STEP-5: LLM INTEGRATION AND ANSWER GENERATION

In [10]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [13]:
user_query = "By 2020, how many employees were there?"
rel_chunks = chroma_db_con.similarity_search(user_query, k=3)

rel_chunks_content = []
for i, chunk in enumerate(rel_chunks):
    rel_chunks_content.append(chunk.page_content)
llm.invoke(f"{user_query}, Use the following context to answer the question: {str(rel_chunks_content)}")
           
    

AIMessage(content='By 2020, the organization had more than fifteen employees.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 81, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSd1oUEXpzUIHIMSNmg4r1LhqTSGM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--e876efae-705d-42ad-adbc-72a90fe41d38-0', usage_metadata={'input_tokens': 81, 'output_tokens': 13, 'total_tokens': 94, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})